In [0]:
SELECT *
FROM customer_shopping_behavior;

-- Q1. What is the total revenue generated by male vs. female customers?
-- Male: 157890, Female: 75191

SELECT gender, SUM(purchase_amount) AS total_revenue
FROM customer_shopping_behavior
GROUP BY gender;

-- Q2. Which customers used a discount but still spent more than the average purchase amount?

SELECT customer_id, purchase_amount
FROM customer_shopping_behavior
WHERE discount_applied = 'Yes' AND purchase_amount > (SELECT AVG(purchase_amount) FROM customer_shopping_behavior);

-- Q3. Which are the top 5 products with the highest average review rating?
-- Gloves, Sandals, Boots, Hat, Skirt
SELECT item_purchased, AVG(review_rating) AS avg_rating
FROM customer_shopping_behavior
GROUP BY item_purchased
ORDER BY avg_rating DESC
LIMIT 5;

-- Q4. Compare the average purchase amounts between standard and express shipping.
SELECT AVG(purchase_amount) AS avg_purchase_amount, shipping_type
FROM customer_shopping_behavior
WHERE shipping_type in ("Standard", "Express")
GROUP BY shipping_type
ORDER BY avg_purchase_amount DESC;

--  Q5. Do subscribed customers spend more? Compare average spend and total revenue between subscribers and non-subscribers
SELECT subscription_status, AVG(purchase_amount) AS avg_purchase_amount, SUM(purchase_amount) AS total_revenue, COUNT(customer_id) as total_customers
FROM customer_shopping_behavior
GROUP BY subscription_status
ORDER BY total_revenue, avg_purchase_amount DESC;

-- Q6. Which 5 products have the highest percentage of purchases with discounts applied?
-- Hat, Sneakers, Coat, Sweater, Pants
SELECT item_purchased, COUNT(*) as total_purchases, SUM(CASE WHEN discount_applied = 'Yes' THEN 1 ELSE 0 END) as discounted_purchases, (SUM(CASE WHEN discount_applied = 'Yes' THEN 1 ELSE 0 END) * 100.0 / COUNT(*)) as discount_percentage
FROM customer_shopping_behavior
GROUP BY item_purchased
ORDER BY discount_percentage DESC
LIMIT 5;

-- Q7. Segment customers into New, Returning, and Loyal based on their total number of previous purchases, and show the count of each segment.
-- Loyal: 3116, Returning: 701, New: 83
WITH customer_type AS (
    SELECT customer_id, previous_purchases, 
    CASE 
        WHEN previous_purchases = 1 THEN 'New'
        WHEN previous_purchases BETWEEN 2 AND 10 THEN 'Returning'
        ELSE 'Loyal'
    END AS customer_segment
    FROM customer_shopping_behavior
)

SELECT customer_segment, COUNT(*) AS `Number of Customers`
FROM customer_type
GROUP BY customer_segment;

-- Q8. What are the top 3 most purchased products within each category?
-- 

WITH item_counts AS (
    SELECT category, item_purchased, COUNT(customer_id) AS total_orders,
    ROW_NUMBER() OVER(PARTITION BY category ORDER BY count(customer_id) DESC ) AS item_rank
    FROM customer_shopping_behavior
    GROUP BY category, item_purchased
)

SELECT item_rank, category, item_purchased, total_orders
FROM item_counts
WHERE item_rank <= 3;

-- Q9. Are customers who are repeat buyers (more than 5 previous purchases) also likely to subscribe?
SELECT subscription_status, COUNT(customer_id) AS repeat_buyers
FROM customer_shopping_behavior
WHERE previous_purchases > 5
GROUP BY subscription_status;

-- Q10. What is the revenue contribution of each age group?
SELECT age_group, SUM(purchase_amount) AS total_revenue
FROM customer_shopping_behavior
GROUP BY age_group
ORDER BY total_revenue DESC;

